In [1]:
# Import libraries for text processing,
# dimensionality reduction, classification, and validation

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

# Load the preprocessed and reduced dataset
# This is a subset of the Sentiment140 corpus with texts and labels

df_reduced = pd.read_csv('twitter_reduced_Dataset.csv', encoding='latin-1')

# Split the predictor variables (texts) and the target variable (sentiment)
X = df_reduced['text'].values
y = df_reduced['target'].values

# Define a custom stopword list
# These words will be filtered during vectorization because they do not provide relevant information

custom_stopwords = [
    'as', 'an', 'the', 'in', 'on', 'at', 'to', 'of', 'and', 'or',
    'is', 'it', 'for', 'with', 'that', 'this', 'was', 'be',
    'are', 'were', 'been', 'from', 'by', 'about', 'into', 'out',
    'up', 'down', 'over', 'under', 'then', 'than', 'so', 'but', 'not'
]



In [2]:
i=0
# Definition of the parameters for exploration through grid search
# components_k: number of latent components to retain with SVD (LSA)
# C_values: values of the regularization parameter for logistic regression

components_k = [100]  # number of latent components for LSA (Truncated SVD)
C_values = [1]        # regularization strength for logistic regression

# Initialize the list to store the results for each combination
results = []

# Cross-validation configuration (5 folds)
# This technique preserves the proportional distribution of classes in each fold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Main loop over all combinations of k (LSA components) and C (regularization)

for k in components_k:
    for C in C_values:
        scores = []  # List to store the AUC-ROC of each fold

        # Cross-validation
        for train_idx, test_idx in cv.split(X, y):
            # Split texts and labels into training and test sets
            texts_train = X[train_idx]
            texts_test = X[test_idx]
            y_train = y[train_idx]
            y_test = y[test_idx]

            # Text vectorization with TF-IDF
            # max_features: maximum number of words selected by importance
            # stop_words: list of stopwords to remove
            # It is fitted only on the training set to avoid leakage

            vectorizer = TfidfVectorizer(max_features=10000, stop_words=custom_stopwords)
            X_train_tfidf = vectorizer.fit_transform(texts_train)
            X_test_tfidf = vectorizer.transform(texts_test)

            # Dimensionality reduction with Truncated SVD (LSA)
            # Transforms TF-IDF vectors into a more compact thematic representation

            svd = TruncatedSVD(n_components=k, random_state=42)
            X_train_lsa = svd.fit_transform(X_train_tfidf)
            X_test_lsa = svd.transform(X_test_tfidf)

            # Train a logistic regression model
            # penalty='l2': ridge regularization
            # C: inverse of the regularization strength (smaller value → more regularization)

            model = LogisticRegression(
                penalty='l2',
                C=C,
                solver='lbfgs',
                max_iter=100,
                random_state=42,
            )
            model.fit(X_train_lsa, y_train)
            print("Iteration: "+str(i) )  # Print to indicate iteration progress

            # Predict probabilities for the positive class
            # and compute the AUC-ROC metric for the current fold

            probs = model.predict_proba(X_test_lsa)[:, 1]
            auc = roc_auc_score(y_test, probs)
            scores.append(auc)
            print("AUC-ROC: Fold " + str(i) +  ":" + str(auc))  # Print the AUC score for the current fold

        # Store the mean results for the current combination (k, C)

        results.append({
            'k_components': k,
            'C': C,
            'mean_auc_roc': np.mean(scores)
        })


Iteration: 0
AUC-ROC: Fold 0:0.757810546875
Iteration: 0
AUC-ROC: Fold 0:0.7696662109375
Iteration: 0
AUC-ROC: Fold 0:0.7638404296875
Iteration: 0
AUC-ROC: Fold 0:0.770311328125
Iteration: 0
AUC-ROC: Fold 0:0.7540312499999999


In [3]:
# Organize the results
df_results = pd.DataFrame(results)

# Display the results table
print("\n Cross-validation results (mean AUC-ROC for k=100):\n")
print(df_results.to_string(index=False, float_format="%.4f"))


 Cross-validation results (mean AUC-ROC for k=100):

 k_components  C  mean_auc_roc
          100  1        0.7631
